# 02 — Recherche documentaire : TF-IDF, BM25 et embeddings

Retrouver une source pertinente est la première étape de l’assistant. Une question
peut reprendre les mots d’un document ou exprimer la même idée avec une autre formulation.
Ce notebook explore trois méthodes de recherche pour observer ces deux situations.

**Objectif :** comprendre comment chaque méthode représente les textes, classe les
documents et répond aux formulations des conseillers.

Le parcours présente le prétraitement lexical, construit une première recherche
TF-IDF, examine des questions annotées et hors périmètre, puis introduit BM25 et les
embeddings. Il se termine par une comparaison des temps d’initialisation et de recherche.
L’évaluation quantitative des classements est développée dans le notebook suivant.


In [1]:
import logging

# Message macOS concernant une redirection multiprocessus inutilisée ici.
# Les autres avertissements et erreurs restent visibles.
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").addFilter(
    lambda record: record.getMessage() != "NOTE: Redirects are currently not supported in MacOs."
)

In [2]:
from IPython.display import Markdown, display

from contextiq.ingestion.loader import DataLoader
from contextiq.ingestion.preprocessing import TextPreprocessor
from contextiq.retrieval.tfidf import TfidfRetriever
from contextiq.settings import Settings

# Aucun chemin utilisateur : le chargeur retrouve les données du projet.
settings = Settings.load()
dataset = DataLoader().load()

## 1. Préparer les textes pour la recherche lexicale

TF-IDF et BM25 partagent le même prétraitement : passage en minuscules, retrait des
accents et séparation des mots. Les nombres, le signe `%` et les négations sont
conservés, car ils participent au sens des conditions commerciales.
Les textes originaux restent disponibles pour la lecture des sources.


In [3]:
preprocessor = TextPreprocessor()
text = "Résiliation : ne pas dépasser 150 euros, sans remise de 30 %."
tokens = preprocessor.tokenize(text)

print("TEXTE ORIGINAL")
print(text)
print("\nTOKENS OBTENUS")
print("-" * 40)

for number, token in enumerate(tokens, start=1):
    print(f"{number:2}. {token}")

print("-" * 40)
print(f"Nombre de tokens : {len(tokens)}")

TEXTE ORIGINAL
Résiliation : ne pas dépasser 150 euros, sans remise de 30 %.

TOKENS OBTENUS
----------------------------------------
 1. resiliation
 2. ne
 3. pas
 4. depasser
 5. 150
 6. euros
 7. sans
 8. remise
 9. de
10. 30
11. %
----------------------------------------
Nombre de tokens : 11


## 2. Construire l’index TF-IDF

TF-IDF pondère les mots selon leur fréquence dans les documents et leur rareté dans
le corpus. La similarité cosinus sert ensuite à classer les documents pour une question.
Le vocabulaire et les poids sont appris uniquement sur les documents, sans utiliser
les questions ni leurs annotations.


In [4]:
from time import perf_counter

start = perf_counter()
retriever = TfidfRetriever(dataset.documents)
initialization_seconds = {"TF-IDF": perf_counter() - start}
print(f"Documents indexés : {len(retriever.documents)}")
print(f"Termes du vocabulaire : {len(retriever.vectorizer.vocabulary_)}")

Documents indexés : 6
Termes du vocabulaire : 88


## 3. Observer les résultats sur une question

Une première question permet d’examiner les documents retrouvés, leur ordre et leur
score. Le texte affiché permet de vérifier si la source traite effectivement le sujet.
Le score mesure une proximité lexicale ; il ne représente pas la probabilité que
le contexte permette de répondre correctement.


In [5]:
question = "Quels frais pour une résiliation annuelle ?"
results = retriever.retrieve(question, top_k=settings.retrieval.top_k)

for rank, result in enumerate(results, start=1):
    print(f"{rank}. {result.document.id} — score {result.score:.3f}")
    print(result.document.text)
    print()

1. D2 — score 0.212
La résiliation d'un contrat annuel avant son terme entraîne des frais égaux à 50 % des mensualités restantes, plafonnés à 150 euros.

2. D5 — score 0.123
Une offre de rétention de 3 mois à -30 % peut être proposée uniquement aux clients ayant plus de 12 mois d'ancienneté et sans retard de paiement.

3. D3 — score 0.077
En cas de déménagement, le client peut résilier sans frais son contrat annuel ou biannuel sur présentation d'un justificatif de domicile.



## 4. Examiner les questions annotées

Les documents retrouvés sont confrontés aux sources attendues pour repérer les
réussites et les erreurs de recherche. Les résultats de score nul sont exclus :
la liste peut donc contenir moins de K documents.
Cette inspection prépare la comparaison quantitative sans ajuster les paramètres.


In [6]:
for item in dataset.questions:
    results = retriever.retrieve(item.q, top_k=settings.retrieval.top_k)
    print(item.q)
    print("Sources attendues :", item.gold_docs or "Hors périmètre")
    print("Sources retrouvées :", [(r.document.id, round(r.score, 3)) for r in results])
    print()

Combien de temps de préavis pour un contrat mensuel ?
Sources attendues : ['D1']
Sources retrouvées : [('D1', 0.478), ('D4', 0.318), ('D3', 0.285)]

Quels frais si je quitte mon abonnement annuel avant la fin ?
Sources attendues : ['D2']
Sources retrouvées : [('D2', 0.38), ('D4', 0.205), ('D3', 0.172)]

Je déménage, puis-je partir sans payer ?
Sources attendues : ['D3']
Sources retrouvées : [('D3', 0.174), ('D6', 0.174), ('D5', 0.134)]

Ma ligne est coupée, que faire ?
Sources attendues : ['D4']
Sources retrouvées : [('D1', 0.17), ('D4', 0.134)]

Puis-je proposer une remise à un client qui veut partir ?
Sources attendues : ['D5']
Sources retrouvées : [('D1', 0.314), ('D2', 0.2), ('D3', 0.179)]

Quel est le cours de l'action de l'entreprise ?
Sources attendues : Hors périmètre
Sources retrouvées : [('D4', 0.452), ('D5', 0.311), ('D3', 0.298)]



## 5. Observer le comportement hors périmètre

Des mots courants peuvent conduire à retrouver des documents alors que la question
n’a pas de réponse dans la base. À l’inverse, une question pertinente peut partager
peu de mots avec sa source. Cette observation distingue la recherche de similarités
de la décision d’accepter ou de refuser une question.


In [7]:
for item in dataset.questions:
    if not item.gold_docs:
        results = retriever.retrieve(item.q, top_k=settings.retrieval.top_k)
        print("Question hors périmètre :", item.q)
        print("Résultats lexicaux :", [(r.document.id, round(r.score, 3)) for r in results])

Question hors périmètre : Quel est le cours de l'action de l'entreprise ?
Résultats lexicaux : [('D4', 0.452), ('D5', 0.311), ('D3', 0.298)]


## 6. Introduire BM25 et la recherche sémantique

BM25 limite l’effet des répétitions et tient compte de la longueur des documents.
Le paramètre `k1` contrôle la saturation des fréquences et `b` la normalisation des
longueurs. L’IDF utilisé est `log(1 + (N - df + 0.5) / (df + 0.5))` ; les termes
répétés dans la question sont comptés une seule fois.

Les embeddings représentent les textes par des vecteurs qui permettent de comparer
leur proximité sémantique. Le modèle multilingue `paraphrase-multilingual-MiniLM-L12-v2`
reçoit les textes originaux et utilise son propre tokenizer, contrairement aux
méthodes lexicales. Les trois méthodes indexent ici les documents entiers et
n’utilisent pas les annotations pour apprendre leur représentation.

Le modèle sémantique accepte au maximum 128 tokens par texte. Les textes plus longs
sont refusés afin d’éviter de perdre silencieusement une condition contractuelle.


In [8]:
from contextiq.evaluation.evaluator import RetrievalEvaluator
from contextiq.retrieval.bm25 import BM25Retriever
from contextiq.retrieval.embeddings import EmbeddingRetriever

start = perf_counter()
bm25 = BM25Retriever(dataset.documents, **settings.retrieval.bm25.model_dump())
initialization_seconds["BM25"] = perf_counter() - start

start = perf_counter()
embeddings = EmbeddingRetriever(dataset.documents, **settings.retrieval.embeddings.model_dump())
initialization_seconds["Embeddings"] = perf_counter() - start

retrievers = {"TF-IDF": retriever, "BM25": bm25, "Embeddings": embeddings}
evaluator = RetrievalEvaluator()
print("Modèle :", embeddings.model_name)
print("Révision :", embeddings.revision)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modèle : sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Révision : e8f8c211226b894fcb81acc59f3b34ba3efd5f42


## 7. Comparer les temps d’initialisation et de recherche

Le temps d’initialisation inclut la construction de l’index et, pour les embeddings,
le chargement des poids déjà disponibles. Il exclut le téléchargement et les imports.
La latence moyenne mesure une recherche après échauffement.
Ces mesures sur un petit nombre de questions donnent des ordres de grandeur ; elles
ne constituent pas un test de charge et dépendent de la machine utilisée.


In [9]:
timing_rows = [
    "| Méthode | Initialisation (s) | Latence moyenne après échauffement (ms) |",
    "|---|---:|---:|",
]
for name, engine in retrievers.items():
    engine.retrieve(dataset.questions[0].q, top_k=settings.retrieval.top_k)
    start = perf_counter()
    for item in dataset.questions:
        engine.retrieve(item.q, top_k=settings.retrieval.top_k)
    mean_ms = 1000 * (perf_counter() - start) / len(dataset.questions)
    timing_rows.append(f"| {name} | {initialization_seconds[name]:.3f} | {mean_ms:.3f} |")
display(Markdown("\n".join(timing_rows)))

| Méthode | Initialisation (s) | Latence moyenne après échauffement (ms) |
|---|---:|---:|
| TF-IDF | 0.014 | 0.414 |
| BM25 | 0.001 | 0.017 |
| Embeddings | 0.643 | 34.485 |

## Prolongement de l’analyse

Le [notebook d’évaluation du retrieval](03_retrieval_evaluation.ipynb) compare les
classements à partir des annotations et examine les erreurs question par question.
